# Extension: PDF Parsing with AI in Lakeflow Designer

An extension to the [2. Lakeflow Designer](#notebook-3452153026155251) workshop notebook. It turns Northgate Advisory's Statement of Work PDFs into `contract_terms` and `contracted_rates` tables with `ai_parse_document` and `ai_extract`. It contains the facilitator demo script, a take-home lab, and the same extraction in SQL, with an accuracy check against the answer key.

In [0]:
dbutils.widgets.text("catalog", "praju_stable_2_catalog", "Catalog")
dbutils.widgets.text("schema", "consulting_billing", "Schema")
dbutils.widgets.text("volume", "raw_files", "Volume")

CATALOG = dbutils.widgets.get("catalog")
SCHEMA = dbutils.widgets.get("schema")
VOLUME = dbutils.widgets.get("volume")

FQ_SCHEMA = f"`{CATALOG}`.`{SCHEMA}`"
VOLUME_PATH = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}"

# Fail fast if the setup notebook hasn't been run for this catalog/schema
missing = [t for t in ["employees", "clients", "engagements", "engagement_assignments"]
           if not spark.catalog.tableExists(f"{CATALOG}.{SCHEMA}.{t}")]
if missing:
    raise RuntimeError(f"Missing tables {missing}: run the '1. Workshop Data Setup' notebook first.")

print(f"Schema : {CATALOG}.{SCHEMA}")
print(f"Volume : {VOLUME_PATH}")

In [0]:
# Point the session at the workshop schema and expose the contract PDFs as a temporary view,
# so the SQL cells below can read them by name.
spark.sql(f"USE {FQ_SCHEMA}")

spark.sql(f"""
  CREATE OR REPLACE TEMP VIEW raw_contracts AS
  SELECT _metadata.file_name AS file_name, content
  FROM read_files('{VOLUME_PATH}/contracts/', format => 'binaryFile', pathGlobFilter => '*.pdf')
""")
display(spark.sql("SELECT COUNT(*) AS pdf_files FROM raw_contracts"))

## Lakeflow Designer, from PDF contracts to a table (`ai_parse_document` + `ai_extract`)

**Goal:** the agreed day rates and fee caps only exist in the signed Statement of Work PDFs. Extract them into two tables, `contract_terms` and `contracted_rates`, then use them to value WIP at the **contracted** rate.

**1. Source: the contracts folder**
* New visual data prep `<initials>_sow_extraction`. Add a **Source** on `/Volumes/<catalog>/consulting_billing/raw_files/contracts/`. For PDFs, Designer reads **one row per file**, with the binary content in a column. You should see 17 rows: 16 SOWs and 1 change request.

**2. AI Function: `ai_parse_document`**
* Add an **AI Function** operator, select `ai_parse_document`, and choose the binary content column as the input. Preview the result: the parser keeps the document layout, including headings, key/value tables and the day-rate table as HTML.

**3. AI Function: `ai_extract`**
* Add a second **AI Function** operator, select `ai_extract`, use the parsed output as the input, and define these fields:

| Field | Type | Description |
| --- | --- | --- |
| `document_type` | enum: Statement of Work, Change Request | Type of contract document |
| `document_reference` | string | SOW reference (SOW-26-NNN) or change request reference (CR-26-NNN-NN) |
| `engagement_code` | string | Engagement code in the format ENG-26-NNN |
| `client_name` | string | Client legal name |
| `billing_basis` | enum: Time & Materials, Capped T&M, Fixed Fee | Billing basis |
| `fee_amount_gbp` | number | Fixed fee, fee cap or estimated fees in GBP. For a change request, the revised total |
| `start_date` / `end_date` | string | Dates in YYYY-MM-DD format |
| `purchase_order` | string | Client PO number |
| `payment_terms_days` | integer | Days from invoice |
| `day_rates` | array of objects: `grade` (string), `standard_day_rate_gbp` (number), `discount_pct` (number), `agreed_day_rate_gbp` (number) | Agreed day rate table |

Set the types exactly as listed. If you leave everything as `string`, the fee comes back as text such as "GBP 212,000 (excluding VAT)", and `day_rates` comes back as one block of text whose separators change from PDF to PDF. That makes it impossible to split into rows reliably.

**Setting the types.** The operator stores the fields as a JSON schema inside a single `ai_extract(...)` expression. If the field editor gives you a type picker, set each type there. Otherwise, or if you'd rather paste, replace the operator's expression with this one:

```
ai_extract(`ai_parse_document(content)`, '{
 "document_type": {"type": "enum", "labels": ["Statement of Work", "Change Request"]},
 "document_reference": {"type": "string", "description": "SOW-26-NNN or CR-26-NNN-NN"},
 "engagement_code": {"type": "string", "description": "Format ENG-26-NNN"},
 "client_name": {"type": "string", "description": "Client legal name"},
 "billing_basis": {"type": "enum", "labels": ["Time & Materials", "Capped T&M", "Fixed Fee"]},
 "fee_amount_gbp": {"type": "number", "description": "Fixed fee, fee cap or estimated fees in GBP; for a change request, the revised total"},
 "start_date": {"type": "string", "description": "YYYY-MM-DD"},
 "end_date": {"type": "string", "description": "YYYY-MM-DD"},
 "purchase_order": {"type": "string", "description": "Client PO number"},
 "payment_terms_days": {"type": "integer", "description": "Days from invoice"},
 "day_rates": {"type": "array", "description": "Agreed day rate table", "items": {"type": "object", "properties": {
   "grade": {"type": "string"},
   "standard_day_rate_gbp": {"type": "number"},
   "discount_pct": {"type": "number"},
   "agreed_day_rate_gbp": {"type": "number"}}}}
}')
```

The input column name in backticks must match the name of the `ai_parse_document` output column in your prep.

**4. Flatten the result**

Preview the `ai_extract` output. It is a single VARIANT column (named `ai_extract(ai_parse_document(content))`) with this shape:

`{"error_message": null, "metadata": {...}, "response": {"engagement_code": {"value": "ENG-26-006"}, "day_rates": [{"grade": {"value": "Director"}, "agreed_day_rate_gbp": {"value": 1850}, ...}, ...], ...}}`

Every field sits under `response`, and each value is wrapped in its own `value` key. A plain Parse JSON on the whole column would only give you `error_message`, `metadata` and `response`, so read each field directly instead.

*a. Prepare: one column per field.* Add a **Prepare** operator. For each field, add a **Formula** action, with the target column named after the field. Use this pattern, where `e` stands for `` `ai_extract(ai_parse_document(content))` ``:

| Column | Expression |
| --- | --- |
| `document_type` | `e:response.document_type.value::string` |
| `document_reference` | `e:response.document_reference.value::string` |
| `engagement_code` | `e:response.engagement_code.value::string` |
| `client_name` | `e:response.client_name.value::string` |
| `billing_basis` | `e:response.billing_basis.value::string` |
| `fee_amount_gbp` | `e:response.fee_amount_gbp.value::double` |
| `start_date` / `end_date` | `coalesce(try_to_date(e:response.start_date.value::string, 'yyyy-MM-dd'), try_to_date(e:response.start_date.value::string, 'd MMMM yyyy'))` (same for `end_date`). The PDFs write dates as "14 September 2026", and the model doesn't always reformat them. |
| `purchase_order` | `e:response.purchase_order.value::string` |
| `payment_terms_days` | `e:response.payment_terms_days.value::int` |
| `day_rates` | `e:response.day_rates` (keep it as VARIANT: do **not** cast it to string) |

This gives `contract_terms`: 17 rows, one per document.

*b. SQL: one row per grade.* Add a **SQL** operator after the Prepare step (refer to its input by the upstream operator's name) and explode the array:

`SELECT t.engagement_code, r.value:grade.value::string AS grade, r.value:standard_day_rate_gbp.value::double AS standard_day_rate_gbp, r.value:discount_pct.value::double AS discount_pct, r.value:agreed_day_rate_gbp.value::double AS agreed_day_rate_gbp FROM <prepare_step> t, LATERAL variant_explode(t.day_rates) r WHERE t.document_type = 'Statement of Work'`

This gives `contracted_rates`: **79 rows** (16 SOWs, 4 to 6 grades each). The change request has no rate table, so it drops out.

> **Genie Code prompt:** *The ai_extract column is a VARIANT where each field is under response.<field>.value. Add a Prepare step with one Formula column per field (fee_amount_gbp as double, payment_terms_days as int, start_date and end_date as dates, keep day_rates as VARIANT). Then add a step that explodes day_rates into one row per grade for Statement of Work documents only, with grade, standard_day_rate_gbp, discount_pct and agreed_day_rate_gbp.*

**Checkpoint:** `contracted_rates` should have 79 rows, and `agreed_day_rate_gbp` should be numeric (for example, 1850 for Director on ENG-26-006). Far fewer rows, or rates stored as text like "1,715", means the `ai_extract` types in step 3 were left as `string`.

**5. Two outputs**
* `<initials>_contract_terms`: one row per document
* `<initials>_contracted_rates`: one row per engagement and grade, Statement of Work documents only

**6. Use the contracted rates in the Part 1 flow**
* Back in `<initials>_timesheet_wip` (built in Part 1 of the main notebook), add a **Left join** to `<initials>_contracted_rates` on `engagement_code` + `grade`. Add `wip_contract_gbp` = `days * agreed_day_rate_gbp` and `rate_realisation` = `wip_contract_gbp / wip_standard_gbp`.

**Discussion points**
* Some timesheet rows get no contracted rate (a null after the left join). These are people booking time to engagements they aren't staffed on, at a grade the SOW doesn't price: a real billing risk.
* The change request for ENG-26-005 raises the fee cap. Which document should win when you calculate fee burn?
* AI output is non-deterministic: save the extraction to a table once rather than re-extracting on every run, and check it against the source (see the accuracy check at the end of this notebook).

---

### Demo script (facilitator, about 12 minutes)

This is a facilitator-led demo using a pre-built visual data prep. Attendees watch, and don't build it themselves. Their hands-on time stays in Parts 1 to 3 of the main [2. Lakeflow Designer](#notebook-3452153026155251) notebook. The demo closes Workshop 1 just before the break, and leads into the AI sessions that follow.

**Before the session (once, about 10 minutes)**
1. In **Catalog Explorer**, open the volume `raw_files` > `_facilitator` and download `sow_extraction_demo.designer.ipynb`.
2. In the workspace, open this folder and click **⋮ > Import**. Choose **File** and upload the downloaded file. Rename it `sow_extraction_demo` if needed.
3. Open the prep and click **Run** once. It writes `designer_contract_terms` (17 rows) and `designer_contracted_rates` (79 rows). It does **not** overwrite the curated `contract_terms` and `contracted_rates` tables that Workshop 2 uses.
4. Preview each operator once, so the AI results are cached and show up instantly during the demo.
5. Have one SOW open in another tab, for example `raw_files/contracts/SOW-26-006.pdf`, scrolled to the day-rate table.
6. Check that your Part 1 prep `<initials>_timesheet_wip` runs, so you can join the rates into it at the end.

**What the prep contains** (left to right)

| # | Operator | What it does | Rows |
| --- | --- | --- | --- |
| 1 | SOW PDFs (Source) | Reads `raw_files/contracts/`, one row per PDF | 17 |
| 2 | Parse PDFs (AI Function) | `ai_parse_document`: text, headings and tables, with layout kept | 17 |
| 3 | Extract contract fields (AI Function) | `ai_extract` with a typed schema (enums, numbers, the day-rate table as a list) | 17 |
| 4 | Flatten contract fields (Prepare) | One Formula action per field: typed columns | 17 |
| 5 | Contract terms (SQL) | One clean row per document | 17 |
| 6 | Contracted rates (SQL) | One row per engagement and grade, SOWs only | 79 |
| 7, 8 | Outputs | `designer_contract_terms`, `designer_contracted_rates` | |

**Talk track**

| Time | Show | Say |
| --- | --- | --- |
| 0:00 | The SOW PDF, at the day-rate table | "The agreed day rates and fee caps only exist in signed PDFs. Today someone retypes these into a spreadsheet. Let's see if Designer can read them for us." |
| 1:00 | Operator 1, preview | "One row per file. 16 Statements of Work and 1 change request. The PDF itself is just a column." |
| 2:00 | Operator 2, preview: click one parsed value | "`ai_parse_document` reads the PDF the way a person does: headings, the key terms table, the rate table. It runs inside Databricks, and the files and results are governed by Unity Catalog." |
| 3:30 | Operator 3, configuration | "Now we tell the AI what we want, like a form: document type, client, fee, payment terms, and the rate table as a list. We say what *kind* of value each field is: a number, a date, or one of a fixed list." |
| 5:00 | Operator 4, preview | "Now each field is a proper column. The fee is a number, so we can add it up, compare it and chart it." Point at `fee_amount_gbp` and `billing_basis`. |
| 6:00 | Operator 6, preview | "One row per grade per engagement: 79 rows from 16 contracts, in seconds. Spot-check against the PDF: SOW-26-006, Director, 1,850." Filter the preview to `ENG-26-006`. |
| 7:30 | Operator 5, filter to ENG-26-005 | "Two documents for the same engagement: the SOW says 212,000, the change request says 257,000. Which one should count? The data shows it, but someone has to decide the rule." |
| 8:30 | Switch to `<initials>_timesheet_wip` | Add a **Join** (left) to table `contracted_rates` on `engagement_code` and `grade`, plus a Formula `wip_contract_gbp = days * agreed_day_rate_gbp`. "Now WIP is valued at what the client actually agreed to pay, not at list price." |
| 10:00 | The join preview, rows with null `agreed_day_rate_gbp` | "These rows have no contracted rate: someone booked time at a grade the SOW doesn't price. That's a real billing risk, found in two minutes." |
| 11:00 | Close | "AI output always needs checking. We saved it to a table once and checked it against the source documents: 79 out of 79 rates correct. After the break, in Workshop 2, we'll do the same with Agent Bricks, with no flow at all." |

**If something goes wrong**
* *A preview is slow or times out:* the AI operators process 17 PDFs. Show the output tables `designer_contract_terms` and `designer_contracted_rates` from the pre-run instead.
* *The values differ from the answer key:* AI output can vary between runs. Re-run the accuracy check at the end of this notebook on the `designer_*` tables (change the table names in that cell), or use the curated `contract_terms` and `contracted_rates` tables, which have been verified.
* *The join in step 8:30 fails:* use the curated `contracted_rates` table (79 rows), not `designer_contracted_rates`, so the demo doesn't depend on the live run.

**Optional, for technical attendees:** open the table of contents and view the generated code behind operator 3, or show how a schema typed as all `string` returns `fee_amount_gbp` as "GBP 212,000 (excluding VAT)". Steps 1-6 above remain as a take-home lab.

## The same extraction in a notebook (answer key)

The cells below do the PDF extraction in SQL and write `nb_*` tables, which act as the **answer key** for this extension. Run them in order after the two setup cells at the top.

* *SQL: Parse and Extract SOW PDFs* calls the AI functions on all 17 PDFs. It is slow, costs money on every run, and overwrites the answer-key tables with a fresh extraction, so only re-run it when you mean to.
* *SQL: Fee Burn by Engagement* also reads `nb_engagement_monthly_billing`, which Part 4 of the main notebook creates. Run that first.
* *Check AI Extraction Accuracy* compares the extraction with the answer key that the [1. Workshop Data Setup](#notebook-3120290690157533) notebook copies to `_facilitator/`. To check the demo prep, point it at `designer_contract_terms` and `designer_contracted_rates` instead of the `nb_*` tables.

In [0]:
%sql
-- Parse each PDF once, extract the fields once, and store the result
CREATE OR REPLACE TABLE nb_contract_documents AS
WITH parsed AS (
  SELECT file_name, ai_parse_document(content, MAP('version', '2.0')) AS parsed
  FROM raw_contracts
)
SELECT
  file_name,
  parsed,
  ai_extract(
    parsed,
    '{
      "document_type":      {"type": "enum", "labels": ["Statement of Work", "Change Request"]},
      "document_reference": {"type": "string", "description": "SOW reference (SOW-26-NNN) or change request reference (CR-26-NNN-NN)"},
      "engagement_code":    {"type": "string", "description": "Engagement code in the format ENG-26-NNN"},
      "client_name":        {"type": "string", "description": "Client legal name"},
      "billing_basis":      {"type": "enum", "labels": ["Time & Materials", "Capped T&M", "Fixed Fee"]},
      "fee_amount_gbp":     {"type": "number", "description": "Fixed fee, fee cap or estimated fees in GBP excluding VAT. For a change request, the revised total fees"},
      "start_date":         {"type": "string", "description": "Service start date, YYYY-MM-DD"},
      "end_date":           {"type": "string", "description": "Service end date, YYYY-MM-DD"},
      "purchase_order":     {"type": "string", "description": "Client purchase order number"},
      "payment_terms_days": {"type": "integer", "description": "Payment terms in days from invoice"},
      "day_rates": {
        "type": "array",
        "description": "Rows of the agreed day rate table",
        "items": {
          "type": "object",
          "properties": {
            "grade":                 {"type": "string"},
            "standard_day_rate_gbp": {"type": "number"},
            "discount_pct":          {"type": "number", "description": "Discount as a percentage, e.g. 12 for 12%"},
            "agreed_day_rate_gbp":   {"type": "number"}
          }
        }
      }
    }',
    MAP('version', '2.1', 'instructions', 'These are Statements of Work and change requests from a professional services firm. Amounts are in GBP.')
  ) AS extracted
FROM parsed;

SELECT file_name, extracted:response:document_type:value::string AS document_type, extracted:error_message AS error
FROM nb_contract_documents ORDER BY file_name;

In [0]:
%sql
-- Flatten the extraction: one row per document, and one row per engagement and grade
CREATE OR REPLACE TABLE nb_contract_terms AS
SELECT
  file_name,
  extracted:response:document_type:value::string                    AS document_type,
  extracted:response:document_reference:value::string               AS document_reference,
  upper(trim(extracted:response:engagement_code:value::string))     AS engagement_code,
  extracted:response:client_name:value::string                      AS client_name,
  extracted:response:billing_basis:value::string                    AS billing_basis,
  extracted:response:fee_amount_gbp:value::double                   AS fee_amount_gbp,
  try_to_date(extracted:response:start_date:value::string)          AS start_date,
  try_to_date(extracted:response:end_date:value::string)            AS end_date,
  extracted:response:purchase_order:value::string                   AS purchase_order,
  extracted:response:payment_terms_days:value::int                  AS payment_terms_days
FROM nb_contract_documents;

CREATE OR REPLACE TABLE nb_contracted_rates AS
SELECT
  upper(trim(d.extracted:response:engagement_code:value::string)) AS engagement_code,
  r.value:grade:value::string                                     AS grade,
  r.value:standard_day_rate_gbp:value::double                     AS standard_day_rate_gbp,
  r.value:discount_pct:value::double                              AS discount_pct,
  r.value:agreed_day_rate_gbp:value::double                       AS agreed_day_rate_gbp
FROM nb_contract_documents d,
  LATERAL variant_explode(d.extracted:response:day_rates) AS r
WHERE d.extracted:response:document_type:value::string = 'Statement of Work';

SELECT * FROM nb_contracted_rates ORDER BY engagement_code, standard_day_rate_gbp DESC;

In [0]:
%sql
-- Fee burn: WIP at contracted rates compared with the current fee (a change request overrides the original SOW)
CREATE OR REPLACE TABLE nb_engagement_fee_burn AS
WITH current_fee AS (
  SELECT engagement_code,
         max_by(fee_amount_gbp, CASE document_type WHEN 'Change Request' THEN 2 ELSE 1 END) AS current_fee_gbp
  FROM nb_contract_terms
  GROUP BY engagement_code
)
SELECT
  b.engagement_code,
  b.engagement_name,
  b.client_name,
  b.billing_type,
  SUM(b.wip_contract_gbp)                                  AS wip_contract_to_date_gbp,
  f.current_fee_gbp,
  ROUND(SUM(b.wip_contract_gbp) / f.current_fee_gbp, 3)    AS fee_burn,
  CASE WHEN SUM(b.wip_contract_gbp) > f.current_fee_gbp       THEN 'Over fee'
       WHEN SUM(b.wip_contract_gbp) > 0.9 * f.current_fee_gbp THEN 'At risk'
       ELSE 'On track' END                                  AS fee_status
FROM nb_engagement_monthly_billing b
JOIN current_fee f USING (engagement_code)
GROUP BY ALL;

-- Note: timesheets cover Aug-Oct 2026 only, so engagements that started earlier show lower burn
SELECT * FROM nb_engagement_fee_burn ORDER BY fee_burn DESC;

In [0]:
# Compare the AI-extracted values with the answer key shipped with the PDFs (copied to _facilitator/ by the setup notebook)
from pyspark.sql import functions as F

truth_terms = spark.read.json(f"{VOLUME_PATH}/_facilitator/sow_terms_answer_key.json").select(
    "engagement_code",
    F.col("fee_gbp").cast("double").alias("true_fee_gbp"),
    F.col("purchase_order").alias("true_po"),
    F.col("payment_terms_days").cast("int").alias("true_payment_terms"))
truth_rates = spark.read.json(f"{VOLUME_PATH}/_facilitator/sow_rates_answer_key.json").select(
    "engagement_code", "grade", F.col("agreed_day_rate_gbp").cast("double").alias("true_agreed_rate"))

terms = spark.table("nb_contract_terms").where("document_type = 'Statement of Work'")
term_check = truth_terms.join(terms, "engagement_code", "left").select(
    "engagement_code",
    (F.col("fee_amount_gbp") == F.col("true_fee_gbp")).alias("fee_ok"),
    (F.col("purchase_order") == F.col("true_po")).alias("po_ok"),
    (F.col("payment_terms_days") == F.col("true_payment_terms")).alias("terms_ok"))
rate_check = truth_rates.join(spark.table("nb_contracted_rates"), ["engagement_code", "grade"], "full_outer") \
    .select((F.col("agreed_day_rate_gbp") == F.col("true_agreed_rate")).alias("rate_ok"))

summary = term_check.agg(
    F.count("*").alias("sows"),
    F.sum(F.col("fee_ok").cast("int")).alias("fees_correct"),
    F.sum(F.col("po_ok").cast("int")).alias("pos_correct"),
    F.sum(F.col("terms_ok").cast("int")).alias("payment_terms_correct")
).crossJoin(rate_check.agg(
    F.count("*").alias("rate_rows"),
    F.sum(F.coalesce(F.col("rate_ok"), F.lit(False)).cast("int")).alias("rates_correct")))
display(summary)

## Facilitator answer key (PDF extraction)

| Check | Expected value |
| --- | --- |
| SOW documents / contracted rate rows | 16 / 79 |
| AI extraction accuracy (fee, PO, payment terms, day rates) | 16/16, 16/16, 16/16, 79/79 |
| ENG-26-005 fee cap: SOW, then after the change request | GBP 212,000, then GBP 257,000 |
| Engagements over fee | ENG-26-003 (109%), ENG-26-007 (109%), ENG-26-010 (101%) |

The AI extraction is non-deterministic, so re-runs can differ slightly. Re-run the accuracy check after every extraction.